# LeetCode 652 — Find Duplicate Subtrees bằng Python

Link đề: https://leetcode.com/problems/find-duplicate-subtrees/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Hash Table, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def findDuplicateSubtrees(self, root: Optional[TreeNode]) -> List[Optional[TreeNode]]:
        ...
```

Ghi chú: bài này dạy kỹ thuật **gán "mã định danh" (ID) cho từng dạng cây con** để so sánh hai cây con trong `O(1)`. Notebook đi từ so sánh từng cặp, qua tuần tự hóa thành chuỗi, đến gán ID (đệ quy, hậu thứ tự lặp, và duyệt BFS ngược). Vì kết quả là **các nút cây** và thứ tự trả về không quan trọng, ta tự dựng `TreeNode`, hàm dựng cây từ danh sách kiểu LeetCode, và so sánh kết quả sau khi tuần tự hóa rồi sắp xếp.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho cây nhị phân, trả về **một đại diện** (nút gốc) cho **mỗi kiểu** cây con xuất hiện từ 2 lần trở lên (cùng cấu trúc và cùng giá trị).
2. Nắm các công cụ Python: lớp `TreeNode`, dựng cây từ danh sách theo thứ tự từng tầng (`None` = không có nút), `collections.Counter`/`dict` với khóa là `tuple`, và duyệt cây bằng ngăn xếp thay cho đệ quy.
3. Đi từ **so sánh từng cặp cây con** → **tuần tự hóa thành chuỗi** → **gán ID `(val, id_trái, id_phải)`** (đệ quy, hậu thứ tự lặp, BFS ngược).
4. Hiểu mấu chốt: hai cây con trùng nhau **khi và chỉ khi** bộ ba `(giá trị, dạng cây con trái, dạng cây con phải)` trùng nhau; nếu "dạng" được đánh một số nguyên thì so sánh chỉ tốn `O(1)`.
5. Tránh các lỗi kinh điển: tuần tự hóa thiếu dấu `#` cho nút rỗng hoặc thiếu dấu phân cách, so sánh nông (chỉ giá trị con trực tiếp), trả cùng một kiểu nhiều lần (đếm `>= 2` thay vì `== 2`), và đệ quy quá sâu với cây lệch 5000 nút.

## 2. Đề bài & ví dụ

Cho `root` là gốc của một cây nhị phân, trả về tất cả các **cây con trùng nhau** (duplicate subtrees).

Với **mỗi kiểu** cây con trùng nhau, bạn chỉ cần trả về **nút gốc của một trong số chúng**.

Hai cây là **trùng nhau** nếu có **cùng cấu trúc** và **cùng giá trị** tại các nút tương ứng.

**Ví dụ 1**

- Input: `root = [1,2,3,4,null,2,4,null,null,4]`
- Output: `[[2,4],[4]]`

**Ví dụ 2**

- Input: `root = [2,1,1]`
- Output: `[[1]]`

**Ví dụ 3**

- Input: `root = [2,2,2,3,null,3,null]`
- Output: `[[2,3],[3]]`

(Cây được cho dưới dạng danh sách theo **từng tầng, từ trái sang phải**, `null` là chỗ không có nút. Mỗi phần tử của output là cây con được viết cũng theo cách đó. Thứ tự các cây con trong output không quan trọng.)

**Ràng buộc:**

- Số nút trong cây nằm trong khoảng `[1, 5000]`
- `-200 <= Node.val <= 200`

Hệ quả: cây có thể **lệch** (như một danh sách liên kết) sâu tới 5000 tầng, vượt giới hạn đệ quy mặc định của Python (khoảng 1000); và giá trị chỉ có 401 khả năng nên cây ngẫu nhiên có rất nhiều cây con trùng nhau.

In [1]:
from typing import List, Optional
from collections import Counter, deque
import json
import random
import sys
import time


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"


def build_tree(vals):
    """Dựng cây từ danh sách theo từng tầng (kiểu LeetCode). None = không có nút."""
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root


def to_list(root):
    """Ngược lại: cây -> danh sách theo từng tầng, bỏ các None thừa ở cuối."""
    if root is None:
        return []
    out = []
    q = deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            q.append(node.left)
            q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


ex1 = [1, 2, 3, 4, None, 2, 4, None, None, 4]
ex2 = [2, 1, 1]
ex3 = [2, 2, 2, 3, None, 3, None]
for name, vals in [("Ví dụ 1", ex1), ("Ví dụ 2", ex2), ("Ví dụ 3", ex3)]:
    r = build_tree(vals)
    print(name, vals, "-> dựng lại:", to_list(r), "| khớp:", to_list(r) == vals)

Ví dụ 1 [1, 2, 3, 4, None, 2, 4, None, None, 4] -> dựng lại: [1, 2, 3, 4, None, 2, 4, None, None, 4] | khớp: True
Ví dụ 2 [2, 1, 1] -> dựng lại: [2, 1, 1] | khớp: True
Ví dụ 3 [2, 2, 2, 3, None, 3, None] -> dựng lại: [2, 2, 2, 3, None, 3] | khớp: False


## 3. Kiến thức nền

### 3.1. Cây nhị phân, `TreeNode`, và danh sách theo tầng

Mỗi nút có `val`, `left`, `right` (có thể là `None`). **Cây con** của nút `x` là `x` cùng toàn bộ hậu duệ. Danh sách `[1,2,3,4,None,2,4,...]` được đọc theo từng tầng: gốc là `1`; hai phần tử tiếp theo là con trái, con phải của `1`; hai phần tử tiếp theo nữa là con của `2` (nút đầu hàng đợi), rồi con của `3`, ... `None` nghĩa là "chỗ đó không có nút" (và nó **không** có con nên không chiếm thêm chỗ cho tầng sau).

Hàm `build_tree` ở trên dùng một hàng đợi `deque`: lấy nút ra, gán hai phần tử kế tiếp của danh sách làm con trái/phải.

In [2]:
r = build_tree(ex1)
print("gốc:", r.val, "| con trái:", r.left.val, "| con phải:", r.right.val)
print("con của nút 2 (trái):", r.left.left, r.left.right)
print("con của nút 3:", r.right.left, r.right.right)
print("cháu: 2.left =", r.right.left.left, "| 4 (phải của 3) =", r.right.right)

gốc: 1 | con trái: 2 | con phải: 3
con của nút 2 (trái): TreeNode(4) None
con của nút 3: TreeNode(2) TreeNode(4)
cháu: 2.left = TreeNode(4) | 4 (phải của 3) = TreeNode(4)


### 3.2. Duyệt hậu thứ tự (post-order)

Duyệt **hậu thứ tự** thăm con trái, rồi con phải, rồi mới đến chính nút. Nó hợp bài này vì để biết "dạng" của cây con tại một nút ta cần biết dạng của **hai cây con con** trước. Đệ quy: `dfs(node)` gọi `dfs(node.left)`, `dfs(node.right)`, rồi xử lý `node`.

In [3]:
def postorder(node, out):
    if node is None:
        return
    postorder(node.left, out)
    postorder(node.right, out)
    out.append(node.val)


out = []
postorder(build_tree(ex1), out)
print(out)    # [4, 2, 4, 2, 4, 3, 1] : con trước, cha sau

[4, 2, 4, 2, 4, 3, 1]


### 3.3. Tuần tự hóa (serialization) và vì sao cần `#`

**Tuần tự hóa** là biến cây thành một chuỗi sao cho hai cây trùng nhau **khi và chỉ khi** hai chuỗi giống nhau. Cách thường dùng: tiền thứ tự `"giá_trị,trái,phải"` và viết `#` cho nút rỗng. Hai điểm bắt buộc:

- **Dấu `#` cho `None`**: nếu bỏ, cây "có con trái 5" và cây "có con phải 5" cho cùng chuỗi.
- **Dấu phân cách `,`**: nếu bỏ, giá trị `1` rồi `11` và `11` rồi `1` đều ra `"111"`.

In [4]:
def serialize(node):
    if node is None:
        return "#"
    return f"{node.val},{serialize(node.left)},{serialize(node.right)}"


a = build_tree([1, 5])           # 1 có con TRÁI 5
b = build_tree([1, None, 5])     # 1 có con PHẢI 5
print(serialize(a), "|", serialize(b))
print("khác nhau:", serialize(a) != serialize(b))
print("nếu bỏ '#' thì cả hai chỉ còn '1,5':", "1,5" == "1,5")

1,5,#,#,# | 1,#,5,#,#
khác nhau: True
nếu bỏ '#' thì cả hai chỉ còn '1,5': True


### 3.4. `dict`/`Counter` với khóa là `tuple`

`tuple` (bộ giá trị bất biến) dùng được làm khóa của `dict`/`set`. `Counter` đếm số lần xuất hiện. `table.setdefault(key, giá_trị_mới)` trả về giá trị đã có của `key`, hoặc đặt `giá_trị_mới` nếu chưa có — rất hợp để **cấp ID mới** cho một dạng mới.

In [5]:
c = Counter()
for x in ["a", "b", "a", "a"]:
    c[x] += 1
print(c, c["a"], c["z"])           # khóa chưa có -> 0

table = {}
for key in [(4, 0, 0), (2, 1, 0), (4, 0, 0)]:
    ident = table.setdefault(key, len(table) + 1)
    print(key, "-> ID", ident)
print(table)

Counter({'a': 3, 'b': 1}) 3 0
(4, 0, 0) -> ID 1
(2, 1, 0) -> ID 2
(4, 0, 0) -> ID 1
{(4, 0, 0): 1, (2, 1, 0): 2}


### 3.5. Giới hạn đệ quy và duyệt bằng ngăn xếp

Python mặc định chỉ cho đệ quy sâu khoảng 1000 tầng. Cây lệch 5000 nút cần đệ quy sâu 5000 → `RecursionError`. Có hai cách tránh: tăng giới hạn (`sys.setrecursionlimit`), hoặc **viết lại bằng ngăn xếp** (`list` với `append`/`pop`) tự quản lý. Ta sẽ làm cách thứ hai ở cách 4.

In [6]:
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())


def depth(node):
    return 0 if node is None else 1 + max(depth(node.left), depth(node.right))


chain = None
for _ in range(2000):                  # cây lệch trái 2000 nút
    chain = TreeNode(1, chain, None)
try:
    print(depth(chain))
except RecursionError:
    print("RecursionError với cây lệch 2000 nút")

giới hạn đệ quy mặc định: 1000
RecursionError với cây lệch 2000 nút


## 4. Ý tưởng nền tảng / trực giác

Làm tay với ví dụ 2: `[2,1,1]` — gốc `2` có hai con `1` và `1` đều là lá.

```
    2
   / \
  1   1
```

Hai nút `1` là hai cây con có cấu trúc "một nút, không con" và cùng giá trị 1 → **trùng nhau**. Cây con gốc `2` chỉ có một → không trùng. Đáp án: `[[1]]`.

Ví dụ 1: `[1,2,3,4,null,2,4,null,null,4]`

```
        1
       / \
      2   3
     /   / \
    4   2   4
       /
      4
```

Các cây con: lá `4` xuất hiện **ba** lần; cây con `2 -> 4` (nút `2` có con trái `4`) xuất hiện **hai** lần; còn lại (`3`, `1`) mỗi cái một lần. Vậy đáp án có **hai kiểu**: `[2,4]` và `[4]` — mỗi kiểu một đại diện, dù lá `4` xuất hiện ba lần.

Cách người ta làm tự nhiên: liệt kê mọi cây con rồi **xem cặp nào giống nhau**. Ta viết đúng như thế ở cách 1.

In [7]:
def all_nodes(root):
    """Mọi nút của cây (mỗi nút là gốc của một cây con), theo thứ tự tiền thứ tự."""
    out, stack = [], [root] if root else []
    while stack:
        node = stack.pop()
        out.append(node)
        if node.right:
            stack.append(node.right)
        if node.left:
            stack.append(node.left)
    return out


root1 = build_tree(ex1)
for node in all_nodes(root1):
    print(f"cây con gốc {node.val}: {to_list(node)}")

cây con gốc 1: [1, 2, 3, 4, None, 2, 4, None, None, 4]
cây con gốc 2: [2, 4]
cây con gốc 4: [4]
cây con gốc 3: [3, 2, 4, 4]
cây con gốc 2: [2, 4]
cây con gốc 4: [4]
cây con gốc 4: [4]


## 5. Cách 1 — Brute Force: so sánh từng cặp cây con

### Ý tưởng

1. Lấy danh sách mọi nút (`all_nodes`), mỗi nút là gốc của một cây con.
2. Hàm `same(a, b)` so sánh hai cây từng nút một (đệ quy): cả hai rỗng → trùng; một rỗng, một không → khác; giá trị khác → khác; còn lại → so sánh con trái và con phải.
3. Duyệt các nút theo thứ tự; với mỗi nút `b`, tìm nút **đầu tiên** `a` đứng trước mà `same(a, b)`. Nếu có, `a` là đại diện của một kiểu trùng; thêm vào kết quả nếu chưa có.

Vì `a` luôn là **lần xuất hiện đầu tiên** của kiểu đó nên mỗi kiểu được đại diện đúng một lần.

### Code

In [8]:
def same(a, b):
    if a is None and b is None:
        return True
    if a is None or b is None:
        return False
    return a.val == b.val and same(a.left, b.left) and same(a.right, b.right)


def find_dup_pairwise(root):
    nodes = all_nodes(root)
    ans = []
    for idx, b in enumerate(nodes):
        for a in nodes[:idx]:
            if same(a, b):
                if all(a is not x for x in ans):     # so sánh danh tính, không so sánh giá trị
                    ans.append(a)
                break
    return ans


def canon(res):
    """Chuẩn hóa kết quả để so sánh: danh sách đã sắp xếp của các chuỗi tuần tự hóa."""
    return sorted(json.dumps(to_list(r)) for r in res)


for name, vals in [("Ví dụ 1", ex1), ("Ví dụ 2", ex2), ("Ví dụ 3", ex3)]:
    res = find_dup_pairwise(build_tree(vals))
    print(name, "->", [to_list(r) for r in res])

Ví dụ 1 -> [[2, 4], [4]]
Ví dụ 2 -> [[1]]
Ví dụ 3 -> [[2, 3], [3]]


### Độ phức tạp và nút thắt

- Có `n` cây con; số cặp `O(n^2)`, mỗi lần `same` tốn tới `O(n)` → **`O(n^3)` thời gian** tệ nhất, `O(n)` bộ nhớ (ngăn xếp đệ quy). Với `n = 5000` hoàn toàn không khả thi.
- Nút thắt: ta so sánh lại những phần cây **giống nhau lặp đi lặp lại** (mỗi cặp so từ đầu). Nếu mỗi cây con có sẵn một "dấu vân tay" thì chỉ cần so dấu vân tay.

## 6. Ý tưởng thuật toán / bước đột phá

**Dấu vân tay của cây con.** Hai cây con trùng nhau ⇔ `(val, dạng_trái, dạng_phải)` trùng nhau. Vì thế:

1. Duyệt hậu thứ tự để có dấu vân tay của hai con **trước** khi xử lý cha.
2. Ghép thành một **khóa** và dùng bảng băm:
   - khóa dạng **chuỗi** (tuần tự hóa) → cách 2;
   - khóa dạng **bộ ba số nguyên** `(val, id_trái, id_phải)`, với `id` là số nguyên nhỏ cấp cho mỗi dạng mới → cách 3, 4, 5.
3. Đếm số lần mỗi dấu vân tay xuất hiện; **ngay khi đếm chạm đúng 2** thì thêm nút đó vào kết quả (nhờ vậy mỗi kiểu chỉ trả một lần, dù xuất hiện 3, 4, ... lần).

Vì sao dùng ID thay vì chuỗi? Chuỗi của cây con kích thước `s` dài `O(s)`, so sánh/băm tốn `O(s)`; còn `(val, id_trái, id_phải)` luôn là bộ **3 số** nên tốn `O(1)`. Tổng quát: ID là bản "nén" của cả cây con xuống một số nguyên.

In [9]:
# Minh họa: gán ID cho ví dụ 2 = [2, 1, 1] bằng tay (null có ID 0)
table = {}

def new_id(key):
    return table.setdefault(key, len(table) + 1)

leaf_left = new_id((1, 0, 0))      # lá 1 bên trái: dạng mới -> ID 1
leaf_right = new_id((1, 0, 0))     # lá 1 bên phải: dạng ĐÃ có -> vẫn ID 1  => trùng nhau!
root_id = new_id((2, leaf_left, leaf_right))
print("ID lá trái:", leaf_left, "| ID lá phải:", leaf_right, "| ID gốc:", root_id)
print("bảng:", table)

ID lá trái: 1 | ID lá phải: 1 | ID gốc: 2
bảng: {(1, 0, 0): 1, (2, 1, 1): 2}


## 7. Các cách tiếp theo

Tổng cộng 5 cách: (1) so sánh từng cặp (mục 5), (2) tuần tự hóa chuỗi + `Counter`, (3) gán ID đệ quy, (4) gán ID bằng hậu thứ tự lặp (ngăn xếp), (5) gán ID bằng duyệt BFS ngược.

### 7.1. Cách 2 — Tuần tự hóa chuỗi + `Counter`

Giải thích từng đoạn:

1. `dfs(node)` trả về chuỗi của cây con tại `node`: `"#"` nếu rỗng, ngược lại `"val,trái,phải"` ghép từ chuỗi hai con.
2. `counter[v] += 1` đếm số lần mỗi chuỗi xuất hiện.
3. Khi `counter[v] == 2` thì thêm `node` vào `ans` (đúng một lần cho mỗi kiểu).

In [10]:
def find_dup_string(root):
    ans = []
    counter = Counter()

    def dfs(node):
        if node is None:
            return "#"
        v = f"{node.val},{dfs(node.left)},{dfs(node.right)}"
        counter[v] += 1
        if counter[v] == 2:
            ans.append(node)
        return v

    dfs(root)
    return ans


for name, vals in [("Ví dụ 1", ex1), ("Ví dụ 2", ex2), ("Ví dụ 3", ex3)]:
    print(name, "->", [to_list(r) for r in find_dup_string(build_tree(vals))])

Ví dụ 1 -> [[4], [2, 4]]
Ví dụ 2 -> [[1]]
Ví dụ 3 -> [[3], [2, 3]]


In [11]:
# Code nộp LeetCode cho cách 2 (dùng cùng TreeNode và Counter ở trên)
class SolutionString:
    def findDuplicateSubtrees(self, root: Optional[TreeNode]) -> List[Optional[TreeNode]]:
        def dfs(root):
            if root is None:
                return '#'
            v = f'{root.val},{dfs(root.left)},{dfs(root.right)}'
            counter[v] += 1
            if counter[v] == 2:
                ans.append(root)
            return v

        ans = []
        counter = Counter()
        dfs(root)
        return ans


print([to_list(r) for r in SolutionString().findDuplicateSubtrees(build_tree(ex1))])

[[4], [2, 4]]


**Độ phức tạp:** chuỗi của cây con `s` nút dài `O(s)`, tạo ra bằng ghép chuỗi tốn `O(s)`. Tổng chiều dài các chuỗi = tổng kích thước mọi cây con = `O(n^2)` với cây lệch, `O(n log n)` với cây cân bằng. Vậy **thời gian và bộ nhớ `O(n^2)`** tệ nhất. Với `n = 5000` lệch: vài chục triệu ký tự — vẫn chạy được trong thực tế nhưng nặng, và độ sâu đệ quy 5000 cũng là vấn đề (mục 13).

### 7.2. Cách 3 — Gán ID đệ quy `(val, id_trái, id_phải)`

Thay chuỗi bằng số nguyên: `id(None) = 0`; mỗi dạng mới nhận ID mới (số kế tiếp). Khóa là bộ ba số, nên mỗi nút tốn `O(1)`.

1. `dfs(node)` trả về ID của cây con tại `node` (`0` nếu `node` rỗng).
2. `key = (node.val, dfs(node.left), dfs(node.right))`.
3. `ident = table.setdefault(key, len(table) + 1)`: dạng cũ giữ ID cũ, dạng mới nhận số mới.
4. Đếm; khi chạm 2 thì thêm `node` vào kết quả.

In [12]:
def find_dup_id_recursive(root):
    table = {}
    count = {}
    ans = []

    def dfs(node):
        if node is None:
            return 0
        key = (node.val, dfs(node.left), dfs(node.right))
        ident = table.setdefault(key, len(table) + 1)
        count[ident] = count.get(ident, 0) + 1
        if count[ident] == 2:
            ans.append(node)
        return ident

    dfs(root)
    return ans


for name, vals in [("Ví dụ 1", ex1), ("Ví dụ 2", ex2), ("Ví dụ 3", ex3)]:
    print(name, "->", [to_list(r) for r in find_dup_id_recursive(build_tree(vals))])

Ví dụ 1 -> [[4], [2, 4]]
Ví dụ 2 -> [[1]]
Ví dụ 3 -> [[3], [2, 3]]


In [13]:
# Code nộp LeetCode cho cách 3
class SolutionIdRecursive:
    def findDuplicateSubtrees(self, root: Optional[TreeNode]) -> List[Optional[TreeNode]]:
        table = {}
        count = {}
        ans = []

        def dfs(node: Optional[TreeNode]) -> int:
            if node is None:
                return 0
            key = (node.val, dfs(node.left), dfs(node.right))
            ident = table.setdefault(key, len(table) + 1)
            count[ident] = count.get(ident, 0) + 1
            if count[ident] == 2:
                ans.append(node)
            return ident

        dfs(root)
        return ans


print([to_list(r) for r in SolutionIdRecursive().findDuplicateSubtrees(build_tree(ex3))])

[[3], [2, 3]]


**Độ phức tạp:** mỗi nút một lần, mỗi lần `O(1)` (băm một bộ ba số) → **`O(n)` thời gian**, `O(n)` bộ nhớ cho `table`, `count` và ngăn xếp đệ quy. Nhược điểm: độ sâu đệ quy bằng chiều cao cây, tới 5000 → có thể `RecursionError` (xem mục 13).

### 7.3. Cách 4 — Gán ID bằng hậu thứ tự lặp (ngăn xếp), lời giải nộp

Cùng ý tưởng cách 3 nhưng **không đệ quy**, để chịu được cây lệch 5000 nút. Ta giữ:

- `stack`: các nút đang chờ xử lý (nút trên cùng là nút đang xét).
- `sid`: từ điển `nút -> ID`, khởi tạo `sid[None] = 0`. Một nút **đã có trong `sid`** nghĩa là cây con của nó đã được tính xong.

Lặp: lấy nút trên cùng `node`.
1. Nếu con trái tồn tại nhưng chưa có ID → đẩy con trái vào ngăn xếp, quay lại đầu vòng lặp.
2. Tương tự với con phải.
3. Khi cả hai con đã có ID → bỏ `node` khỏi ngăn xếp, tính `key = (val, sid[left], sid[right])`, cấp ID, đếm, và ghi `sid[node]`.

In [14]:
def find_dup_id_iterative(root):
    if root is None:
        return []
    table, count, ans = {}, {}, []
    sid = {None: 0}                  # ID của cây con rỗng là 0
    stack = [root]
    while stack:
        node = stack[-1]
        if node.left is not None and node.left not in sid:
            stack.append(node.left)
            continue
        if node.right is not None and node.right not in sid:
            stack.append(node.right)
            continue
        stack.pop()
        key = (node.val, sid[node.left], sid[node.right])
        ident = table.setdefault(key, len(table) + 1)
        sid[node] = ident
        count[ident] = count.get(ident, 0) + 1
        if count[ident] == 2:
            ans.append(node)
    return ans


for name, vals in [("Ví dụ 1", ex1), ("Ví dụ 2", ex2), ("Ví dụ 3", ex3)]:
    print(name, "->", [to_list(r) for r in find_dup_id_iterative(build_tree(vals))])

Ví dụ 1 -> [[4], [2, 4]]
Ví dụ 2 -> [[1]]
Ví dụ 3 -> [[3], [2, 3]]


In [15]:
# Code nộp LeetCode cho cách 4 (hậu thứ tự lặp)
class SolutionIdIterative:
    def findDuplicateSubtrees(self, root: Optional[TreeNode]) -> List[Optional[TreeNode]]:
        if root is None:
            return []
        table, count, ans = {}, {}, []
        sid = {None: 0}
        stack = [root]
        while stack:
            node = stack[-1]
            if node.left is not None and node.left not in sid:
                stack.append(node.left)
                continue
            if node.right is not None and node.right not in sid:
                stack.append(node.right)
                continue
            stack.pop()
            key = (node.val, sid[node.left], sid[node.right])
            ident = table.setdefault(key, len(table) + 1)
            sid[node] = ident
            count[ident] = count.get(ident, 0) + 1
            if count[ident] == 2:
                ans.append(node)
        return ans


print([to_list(r) for r in SolutionIdIterative().findDuplicateSubtrees(build_tree(ex1))])

[[4], [2, 4]]


**Độ phức tạp:** mỗi nút vào/ra ngăn xếp đúng một lần, xử lý `O(1)` → **`O(n)` thời gian**, `O(n)` bộ nhớ, **không** phụ thuộc giới hạn đệ quy.

### 7.4. Cách 5 — Gán ID bằng BFS ngược (xử lý từ tầng dưới lên)

Một cách khác để bảo đảm "con xử lý trước cha" mà không cần đệ quy hay cờ trạng thái:

1. Duyệt BFS theo từng tầng, ghi nút vào danh sách `order` (cha luôn đứng **trước** con của nó).
2. Xử lý `order` **theo thứ tự ngược**: khi tới một nút thì cả hai con (đứng sau trong `order`) đã có ID.
3. Phần tính khóa, cấp ID, đếm giống hệt cách 4.

Kết quả liệt kê theo thứ tự khác (từ tầng sâu lên) — không sao vì đề không yêu cầu thứ tự.

In [16]:
def find_dup_reverse_bfs(root):
    if root is None:
        return []
    order = [root]
    for node in order:                      # danh sách tự dài ra khi ta append: đây chính là BFS
        if node.left:
            order.append(node.left)
        if node.right:
            order.append(node.right)
    table, count, ans = {}, {}, []
    sid = {None: 0}
    for node in reversed(order):
        key = (node.val, sid[node.left], sid[node.right])
        ident = table.setdefault(key, len(table) + 1)
        sid[node] = ident
        count[ident] = count.get(ident, 0) + 1
        if count[ident] == 2:
            ans.append(node)
    return ans


for name, vals in [("Ví dụ 1", ex1), ("Ví dụ 2", ex2), ("Ví dụ 3", ex3)]:
    print(name, "->", [to_list(r) for r in find_dup_reverse_bfs(build_tree(vals))])

Ví dụ 1 -> [[4], [2, 4]]
Ví dụ 2 -> [[1]]
Ví dụ 3 -> [[3], [2, 3]]


In [17]:
# Code nộp LeetCode cho cách 5
class SolutionReverseBFS:
    def findDuplicateSubtrees(self, root: Optional[TreeNode]) -> List[Optional[TreeNode]]:
        if root is None:
            return []
        order = [root]
        for node in order:
            if node.left:
                order.append(node.left)
            if node.right:
                order.append(node.right)
        table, count, ans = {}, {}, []
        sid = {None: 0}
        for node in reversed(order):
            key = (node.val, sid[node.left], sid[node.right])
            ident = table.setdefault(key, len(table) + 1)
            sid[node] = ident
            count[ident] = count.get(ident, 0) + 1
            if count[ident] == 2:
                ans.append(node)
        return ans


print([to_list(r) for r in SolutionReverseBFS().findDuplicateSubtrees(build_tree(ex3))])

[[3], [2, 3]]


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (`order`, `sid`, `table`), không đệ quy. Điều kiện đúng: trong BFS, con luôn được thêm **sau** cha nên khi duyệt ngược, con luôn được xử lý **trước** cha.

## 8. TRACE từng bước

### 8.1. Ví dụ 2: `[2,1,1]` (gán ID hậu thứ tự)

| Bước | Nút | `key = (val, id_trái, id_phải)` | ID | `count[ID]` | Hành động |
|---|---|---|---|---|---|
| 1 | lá `1` (trái) | `(1, 0, 0)` | 1 (mới) | 1 | chưa trùng |
| 2 | lá `1` (phải) | `(1, 0, 0)` | 1 (đã có) | **2** | thêm vào kết quả |
| 3 | gốc `2` | `(2, 1, 1)` | 2 (mới) | 1 | chưa trùng |

Kết quả: `[[1]]`.

### 8.2. Ví dụ 1: `[1,2,3,4,null,2,4,null,null,4]`

Hậu thứ tự (trái, phải, cha). Ký hiệu: lá `4` ID 1; `2→4` ID 2; `3` ID 3; gốc ID 4.

| Bước | Nút | `key` | ID | `count` | Kết quả |
|---|---|---|---|---|---|
| 1 | `4` dưới `2` trái | `(4,0,0)` | 1 mới | 1 | |
| 2 | `2` trái | `(2,1,0)` | 2 mới | 1 | |
| 3 | `4` dưới `2` phải | `(4,0,0)` | 1 | **2** | thêm lá `4` |
| 4 | `2` phải của `3` | `(2,1,0)` | 2 | **2** | thêm `2→4` |
| 5 | `4` phải của `3` | `(4,0,0)` | 1 | 3 | (đã có, không thêm nữa) |
| 6 | `3` | `(3,2,1)` | 3 mới | 1 | |
| 7 | gốc `1` | `(1,2,3)` | 4 mới | 1 | |

Chú ý bước 5: `count` lên 3 nhưng ta chỉ thêm khi chạm **đúng 2**, nên lá `4` chỉ được trả **một lần**.

Hàm `trace_ids` bên dưới in lại đúng các bước này.

In [18]:
def trace_ids(root):
    table, count, ans = {}, {}, []
    sid = {None: 0}
    stack = [root]
    step = 0
    while stack:
        node = stack[-1]
        if node.left is not None and node.left not in sid:
            stack.append(node.left)
            continue
        if node.right is not None and node.right not in sid:
            stack.append(node.right)
            continue
        stack.pop()
        step += 1
        key = (node.val, sid[node.left], sid[node.right])
        is_new = key not in table
        ident = table.setdefault(key, len(table) + 1)
        sid[node] = ident
        count[ident] = count.get(ident, 0) + 1
        note = ""
        if count[ident] == 2:
            ans.append(node)
            note = "  -> THÊM vào kết quả"
        print(f"bước {step}: nút {node.val} (cây con {to_list(node)}) key={key} "
              f"ID={ident} ({'mới' if is_new else 'đã có'}) count={count[ident]}{note}")
    print("kết quả:", [to_list(r) for r in ans])
    return ans


trace_ids(build_tree(ex2))
print()
trace_ids(build_tree(ex1))

bước 1: nút 1 (cây con [1]) key=(1, 0, 0) ID=1 (mới) count=1
bước 2: nút 1 (cây con [1]) key=(1, 0, 0) ID=1 (đã có) count=2  -> THÊM vào kết quả
bước 3: nút 2 (cây con [2, 1, 1]) key=(2, 1, 1) ID=2 (mới) count=1
kết quả: [[1]]

bước 1: nút 4 (cây con [4]) key=(4, 0, 0) ID=1 (mới) count=1
bước 2: nút 2 (cây con [2, 4]) key=(2, 1, 0) ID=2 (mới) count=1
bước 3: nút 4 (cây con [4]) key=(4, 0, 0) ID=1 (đã có) count=2  -> THÊM vào kết quả
bước 4: nút 2 (cây con [2, 4]) key=(2, 1, 0) ID=2 (đã có) count=2  -> THÊM vào kết quả
bước 5: nút 4 (cây con [4]) key=(4, 0, 0) ID=1 (đã có) count=3
bước 6: nút 3 (cây con [3, 2, 4, 4]) key=(3, 2, 1) ID=3 (mới) count=1
bước 7: nút 1 (cây con [1, 2, 3, 4, None, 2, 4, None, None, 4]) key=(1, 2, 3) ID=4 (mới) count=1
kết quả: [[4], [2, 4]]


[TreeNode(4), TreeNode(2)]

### 8.3. Trace tuần tự hóa chuỗi (cách 2) trên ví dụ 3

`[2,2,2,3,null,3,null]` — cây: gốc `2`, hai con `2`, mỗi con `2` có con trái `3`. Cách 2 in từng chuỗi và bộ đếm.

In [19]:
def trace_strings(root):
    counter = Counter()
    ans = []

    def dfs(node):
        if node is None:
            return "#"
        v = f"{node.val},{dfs(node.left)},{dfs(node.right)}"
        counter[v] += 1
        print(f"nút {node.val}: chuỗi = {v!r:32s} count = {counter[v]}")
        if counter[v] == 2:
            ans.append(node)
            print("   -> trùng lần 2, thêm vào kết quả")
        return v

    dfs(root)
    return ans


res = trace_strings(build_tree(ex3))
print([to_list(r) for r in res])

nút 3: chuỗi = '3,#,#'                          count = 1
nút 2: chuỗi = '2,3,#,#,#'                      count = 1
nút 3: chuỗi = '3,#,#'                          count = 2
   -> trùng lần 2, thêm vào kết quả
nút 2: chuỗi = '2,3,#,#,#'                      count = 2
   -> trùng lần 2, thêm vào kết quả
nút 2: chuỗi = '2,2,3,#,#,#,2,3,#,#,#'          count = 1
[[3], [2, 3]]


## 9. Code nộp LeetCode

Chọn cách 4 (gán ID bằng hậu thứ tự lặp) làm lời giải nộp: `O(n)` thời gian, không phụ thuộc độ sâu đệ quy nên an toàn với cây lệch 5000 nút.

In [20]:
class Solution:
    def findDuplicateSubtrees(self, root: Optional[TreeNode]) -> List[Optional[TreeNode]]:
        if root is None:
            return []
        table, count, ans = {}, {}, []
        sid = {None: 0}                    # nút -> ID dạng cây con; cây rỗng có ID 0
        stack = [root]
        while stack:
            node = stack[-1]
            if node.left is not None and node.left not in sid:
                stack.append(node.left)
                continue
            if node.right is not None and node.right not in sid:
                stack.append(node.right)
                continue
            stack.pop()
            key = (node.val, sid[node.left], sid[node.right])
            ident = table.setdefault(key, len(table) + 1)
            sid[node] = ident
            count[ident] = count.get(ident, 0) + 1
            if count[ident] == 2:
                ans.append(node)
        return ans


sol = Solution()
print([to_list(r) for r in sol.findDuplicateSubtrees(build_tree(ex1))])   # [[4], [2, 4]] (thứ tự không quan trọng)
print([to_list(r) for r in sol.findDuplicateSubtrees(build_tree(ex2))])   # [[1]]
print([to_list(r) for r in sol.findDuplicateSubtrees(build_tree(ex3))])   # [[3], [2, 3]]

[[4], [2, 4]]
[[1]]
[[3], [2, 3]]


## 10. Vì sao đúng

**Bổ đề (ID định danh dạng cây con).** Với hai nút `x`, `y` bất kỳ (kể cả `None`), `sid[x] == sid[y]` khi và chỉ khi cây con của `x` và `y` **trùng nhau**.

*Chứng minh quy nạp theo chiều cao.* Cây rỗng: ID 0, và chỉ `None` có ID 0 (mọi ID dạng khác được cấp từ 1). Giả sử bổ đề đúng cho các cây con thấp hơn. Với `x`, `y` không rỗng:

- Nếu cây con trùng nhau: `x.val == y.val`, hai cây con trái trùng nhau nên (giả thiết quy nạp) cùng ID, tương tự bên phải. Khóa `(val, id_trái, id_phải)` giống nhau → `setdefault` cho cùng ID.
- Nếu `sid[x] == sid[y]`: ID chỉ được cấp theo khóa nên hai khóa phải bằng nhau → `val` bằng nhau và ID hai con trái bằng nhau, ID hai con phải bằng nhau → (quy nạp) hai cây con trái trùng, hai cây con phải trùng → cây trùng nhau.

**Tính đúng của kết quả.** `count[ident]` là số nút có dạng `ident`, tức số cây con thuộc **một kiểu**. Nút bị thêm vào `ans` đúng lúc `count` chạm 2:
- mỗi kiểu có ≥ 2 cây con được thêm **đúng một lần** (không thêm ở lần thứ 3, thứ 4, ...);
- kiểu chỉ có 1 cây con không bao giờ chạm 2 nên không bị thêm.

Vì vậy `ans` chứa đúng một đại diện cho mỗi kiểu cây con trùng nhau. Nút đại diện nằm trong cây gốc (đúng yêu cầu "trả về nút gốc của một trong các cây con").

**Về thứ tự xử lý.** Nút chỉ được tính ID khi cả hai con đã có ID (cách 4 bằng điều kiện `not in sid`; cách 5 bằng thứ tự ngược BFS), nên `sid[node.left]`, `sid[node.right]` luôn có sẵn khi tính khóa.

In [21]:
# Kiểm chứng bổ đề trên một cây ngẫu nhiên: sid[x] == sid[y]  <=>  same(x, y)
def build_sid(root):
    order = [root]
    for node in order:
        if node.left:
            order.append(node.left)
        if node.right:
            order.append(node.right)
    table, sid = {}, {None: 0}
    for node in reversed(order):
        key = (node.val, sid[node.left], sid[node.right])
        sid[node] = table.setdefault(key, len(table) + 1)
    return order, sid


def random_tree(n, rng, vmax=2):
    """Cây nhị phân ngẫu nhiên n nút: chèn dần vào chỗ trống ngẫu nhiên."""
    root = TreeNode(rng.randint(-vmax, vmax))
    slots = [(root, "left"), (root, "right")]
    for _ in range(n - 1):
        idx = rng.randrange(len(slots))
        slots[idx], slots[-1] = slots[-1], slots[idx]
        parent, side = slots.pop()
        child = TreeNode(rng.randint(-vmax, vmax))
        setattr(parent, side, child)
        slots.append((child, "left"))
        slots.append((child, "right"))
    return root


rng = random.Random(10)
tree = random_tree(40, rng, vmax=1)
order, sid = build_sid(tree)
pairs = 0
for i, x in enumerate(order):
    for y in order[i + 1:]:
        assert (sid[x] == sid[y]) == same(x, y)
        pairs += 1
print("đã kiểm", pairs, "cặp nút: sid bằng nhau <=> cây con trùng nhau")

đã kiểm 780 cặp nút: sid bằng nhau <=> cây con trùng nhau


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. So sánh từng cặp | `O(n^3)` tệ nhất | `O(n)` |
| 2. Tuần tự hóa chuỗi | `O(n^2)` tệ nhất (cây lệch) | `O(n^2)` (các chuỗi khóa) |
| 3. ID đệ quy | `O(n)` | `O(n)` + ngăn xếp đệ quy `O(h)` |
| 4. ID hậu thứ tự lặp (nộp) | `O(n)` | `O(n)` |
| 5. ID BFS ngược | `O(n)` | `O(n)` |

(`h` là chiều cao cây, tới `n` với cây lệch.) Cell dưới đo **tổng độ dài các chuỗi khóa** của cách 2 trên cây lệch để thấy tăng trưởng bậc 2, trong khi số khóa của cách ID luôn bằng `n`.

In [22]:
def make_left_chain(n, val=1):
    node = None
    for _ in range(n):
        node = TreeNode(val, node, None)
    return node


def total_key_length(root):
    """Tổng độ dài các chuỗi khóa mà cách 2 phải tạo (duyệt lặp để tránh đệ quy sâu)."""
    order = [root]
    for node in order:
        if node.left:
            order.append(node.left)
        if node.right:
            order.append(node.right)
    s = {None: "#"}
    total = 0
    for node in reversed(order):
        s[node] = f"{node.val},{s[node.left]},{s[node.right]}"
        total += len(s[node])
    return total


prev = None
for n in (200, 400, 800, 1600):
    tl = total_key_length(make_left_chain(n))
    note = "" if prev is None else f"(x{tl / prev:.2f} so với n/2)"
    print(f"cây lệch n={n:4d}: tổng độ dài chuỗi khóa = {tl:8d} {note} | số khóa ID = {n}")
    prev = tl

cây lệch n= 200: tổng độ dài chuỗi khóa =    80600  | số khóa ID = 200
cây lệch n= 400: tổng độ dài chuỗi khóa =   321200 (x3.99 so với n/2) | số khóa ID = 400
cây lệch n= 800: tổng độ dài chuỗi khóa =  1282400 (x3.99 so với n/2) | số khóa ID = 800
cây lệch n=1600: tổng độ dài chuỗi khóa =  5124800 (x4.00 so với n/2) | số khóa ID = 1600


## 12. Bộ test

Chiến lược: (a) ví dụ đề; (b) edge case (1 nút, cây lệch, cây hoàn hảo cùng giá trị, giá trị biên); (c) phản ví dụ cho các cách sai; (d) đối chiếu **tất cả** các cách với oracle (so sánh từng cặp độc lập) trên nhiều cây ngẫu nhiên có seed; (e) cây lớn `n = 5000` (lệch và ngẫu nhiên) cho các cách `O(n)`.

Hàm kiểm tra `check_result` làm đúng ba việc: kết quả phải là **nút của chính cây này**; **không có hai nút trùng kiểu**; và **tập kiểu** khớp với oracle. Nhờ vậy thứ tự trả về không ảnh hưởng.

In [23]:
def kinds_oracle(root):
    """Oracle độc lập (so sánh từng cặp): tập các chuỗi tuần tự hóa của mọi kiểu trùng nhau."""
    return sorted(json.dumps(to_list(r)) for r in find_dup_pairwise(root))


def check_result(root, res, expected_kinds=None):
    nodes = all_nodes(root)
    ids = {id(n) for n in nodes}
    assert all(id(r) in ids for r in res), "kết quả chứa nút không thuộc cây"
    for i, a in enumerate(res):
        for b in res[i + 1:]:
            assert not same(a, b), "cùng một kiểu bị trả về hai lần"
    if expected_kinds is None:
        expected_kinds = kinds_oracle(root)
    got = sorted(json.dumps(to_list(r)) for r in res)
    assert got == expected_kinds, (got, expected_kinds)


solvers = {
    "cặp": find_dup_pairwise,
    "chuỗi": find_dup_string,
    "ID đệ quy": find_dup_id_recursive,
    "ID lặp": find_dup_id_iterative,
    "BFS ngược": find_dup_reverse_bfs,
    "nộp String": lambda r: SolutionString().findDuplicateSubtrees(r),
    "nộp IdRecursive": lambda r: SolutionIdRecursive().findDuplicateSubtrees(r),
    "nộp Iterative": lambda r: SolutionIdIterative().findDuplicateSubtrees(r),
    "nộp ReverseBFS": lambda r: SolutionReverseBFS().findDuplicateSubtrees(r),
    "nộp Solution": lambda r: Solution().findDuplicateSubtrees(r),
}


def check_all(vals, expected_lists, label=""):
    expected = sorted(json.dumps(x) for x in expected_lists)
    for name, f in solvers.items():
        root = build_tree(vals)
        res = f(root)
        got = sorted(json.dumps(to_list(r)) for r in res)
        assert got == expected, (label, name, got, expected)
        check_result(root, res, expected)


# (a) ví dụ đề
check_all(ex1, [[2, 4], [4]], "ví dụ 1")
check_all(ex2, [[1]], "ví dụ 2")
check_all(ex3, [[2, 3], [3]], "ví dụ 3")
print("ví dụ đề: OK")

ví dụ đề: OK

In [24]:
# (b) edge case
check_all([1], [], "một nút")
check_all([1, 1], [], "hai nút")
check_all([1, 1, 1], [[1]], "ba nút cùng giá trị")
check_all([1, 2, 3], [], "ba giá trị khác nhau")
check_all([0, 0, 0, 0, None, None, 0], [[0]], "các lá 0 trùng, 0->0 chỉ một")
check_all([-200, -200, -200], [[-200]], "giá trị biên -200")
check_all([200, 200, 200], [[200]], "giá trị biên 200")

# cây hoàn hảo (mọi nút bằng 0), chiều cao 5 (31 nút): mỗi mức chiều cao 1..4 là một kiểu trùng
perfect = [0] * 31
check_all(perfect, [[0], [0, 0, 0], [0, 0, 0, 0, 0, 0, 0], [0] * 15], "cây hoàn hảo 31 nút")

# cây lệch trái toàn giá trị 1 (mọi cây con kích thước khác nhau) => không có kiểu trùng
chain_vals = [1] + [1, None] * 20
for name, f in solvers.items():
    root = build_tree(chain_vals)
    assert f(root) == [], name
print("edge case: OK")

edge case: OK


In [25]:
# (c) phản ví dụ: các cách sai (phân tích ở mục 13)
def wrong_no_null(root):
    """SAI: tuần tự hóa bỏ dấu '#' của nút rỗng."""
    ans, counter = [], Counter()

    def dfs(node):
        parts = [str(node.val)]
        for child in (node.left, node.right):
            if child is not None:                  # nút rỗng bị bỏ qua: mất thông tin trái/phải
                parts.append(dfs(child))
        v = ",".join(parts)
        counter[v] += 1
        if counter[v] == 2:
            ans.append(node)
        return v

    dfs(root)
    return ans


def wrong_no_sep(root):
    """SAI: ghép chuỗi không có dấu phân cách giữa các số."""
    ans, counter = [], Counter()

    def dfs(node):
        if node is None:
            return "#"
        v = f"{node.val}{dfs(node.left)}{dfs(node.right)}"
        counter[v] += 1
        if counter[v] == 2:
            ans.append(node)
        return v

    dfs(root)
    return ans


def wrong_shallow(root):
    """SAI: khóa chỉ gồm giá trị của nút và hai con trực tiếp."""
    ans, counter = [], Counter()
    for node in all_nodes(root):
        key = (node.val, node.left.val if node.left else None, node.right.val if node.right else None)
        counter[key] += 1
        if counter[key] == 2:
            ans.append(node)
    return ans


def wrong_ge_two(root):
    """SAI: thêm vào kết quả mỗi lần đếm >= 2 (trả cùng một kiểu nhiều lần)."""
    ans, counter = [], Counter()

    def dfs(node):
        if node is None:
            return "#"
        v = f"{node.val},{dfs(node.left)},{dfs(node.right)}"
        counter[v] += 1
        if counter[v] >= 2:
            ans.append(node)
        return v

    dfs(root)
    return ans


tests = {
    "wrong_no_null": [1, 1, 1, 5, None, None, 5],            # 1 có con trái 5, 1 có con phải 5
    "wrong_no_sep": [0, 1, 11, 11, None, 1],                  # 1->11 và 11->1 cùng ra '111###'
    "wrong_shallow": [0, 1, 1, 2, None, 2, None, 9],          # hai nút 2 khác nhau ở cháu
    "wrong_ge_two": ex1,                                      # lá 4 xuất hiện 3 lần
}
for name, vals in tests.items():
    f = globals()[name]
    root = build_tree(vals)
    got = sorted(json.dumps(to_list(r)) for r in f(root))
    right = kinds_oracle(root)
    print(f"{name:14s}: đúng = {right} | bản sai = {got}")
    assert got != right

wrong_no_null : đúng = ['[5]'] | bản sai = ['[1, null, 5]', '[5]']
wrong_no_sep  : đúng = [] | bản sai = ['[11, 1]']
wrong_shallow : đúng = [] | bản sai = ['[1, 2]']
wrong_ge_two  : đúng = ['[2, 4]', '[4]'] | bản sai = ['[2, 4]', '[4]', '[4]']


In [26]:
# (d) đối chiếu mọi cách với oracle trên nhiều cây ngẫu nhiên có seed (giá trị nhỏ => nhiều kiểu trùng)
rng = random.Random(652)
n_dup_kinds = 0
for case in range(600):
    n = rng.randint(1, 45)
    vmax = rng.choice([0, 1, 2, 5])
    tree = random_tree(n, rng, vmax)
    expected = kinds_oracle(tree)
    n_dup_kinds += len(expected)
    for name, f in solvers.items():
        res = f(tree)
        check_result(tree, res, expected)
print("600 cây ngẫu nhiên: mọi cách khớp oracle (tổng số kiểu trùng:", n_dup_kinds, ") OK")

600 cây ngẫu nhiên: mọi cách khớp oracle (tổng số kiểu trùng: 1445 ) OK


In [27]:
# (d2) cây dựng sẵn có trùng lặp: sao chép một cây con T vào nhiều chỗ
def planted_tree(rng):
    sub = random_tree(rng.randint(1, 6), rng, vmax=3)
    base = random_tree(rng.randint(5, 25), rng, vmax=3)
    base_nodes = all_nodes(base)                      # chỉ gắn vào các nút của cây nền, không gắn vào bản sao
    for _ in range(rng.randint(2, 4)):                # gắn bản sao (dựng lại từ danh sách) vào chỗ trống
        copy = build_tree(to_list(sub))
        free = [(x, s) for x in base_nodes for s in ("left", "right") if getattr(x, s) is None]
        if not free:
            break
        parent, side = rng.choice(free)
        setattr(parent, side, copy)
    return base, to_list(sub)


rng = random.Random(77)
for case in range(200):
    tree, sub_list = planted_tree(rng)
    expected = kinds_oracle(tree)
    for name, f in solvers.items():
        check_result(tree, f(tree), expected)
    # bản sao T được gắn >= 2 lần nên kiểu của T chắc chắn nằm trong kết quả
    assert json.dumps(sub_list) in expected, (sub_list, expected)
print("200 cây có cây con sao chép sẵn: OK")

200 cây có cây con sao chép sẵn: OK


In [28]:
# (e) cây lớn n = 5000: chỉ chạy các cách O(n) / không đệ quy sâu, đối chiếu chéo
def kinds_via_ids(root):
    return sorted(json.dumps(to_list(r)) for r in find_dup_id_iterative(root))


rng = random.Random(5000)
big = random_tree(5000, rng, vmax=200)       # |val| <= 200 đúng ràng buộc
for name in ("ID lặp", "BFS ngược", "nộp Iterative", "nộp ReverseBFS", "nộp Solution"):
    t0 = time.perf_counter()
    res = solvers[name](big)
    dt = time.perf_counter() - t0
    print(f"ngẫu nhiên 5000 nút, {name:15s}: {len(res):3d} kiểu trùng, {dt * 1000:6.1f} ms")
    check_result(big, res, kinds_via_ids(big))

# cây lệch 5000 nút: các cách lặp chịu được, kết quả là rỗng (mọi cây con khác kích thước)
chain5000 = make_left_chain(5000)
for name in ("ID lặp", "BFS ngược", "nộp Solution"):
    t0 = time.perf_counter()
    res = solvers[name](chain5000)
    dt = time.perf_counter() - t0
    assert res == []
    print(f"lệch trái 5000 nút, {name:12s}: 0 kiểu trùng, {dt * 1000:6.1f} ms")

# cây zig-zag 5000 nút toàn giá trị 0 xen kẽ trái/phải: cũng không có kiểu trùng
zig = None
side = 0
for _ in range(5000):
    zig = TreeNode(0, zig, None) if side == 0 else TreeNode(0, None, zig)
    side ^= 1
assert Solution().findDuplicateSubtrees(zig) == []
print("zig-zag 5000 nút: OK")

ngẫu nhiên 5000 nút, ID lặp         : 363 kiểu trùng,    5.5 ms
ngẫu nhiên 5000 nút, BFS ngược      : 363 kiểu trùng,    3.7 ms
ngẫu nhiên 5000 nút, nộp Iterative  : 363 kiểu trùng,    2.9 ms


ngẫu nhiên 5000 nút, nộp ReverseBFS : 363 kiểu trùng,    3.2 ms
ngẫu nhiên 5000 nút, nộp Solution   : 363 kiểu trùng,    4.8 ms
lệch trái 5000 nút, ID lặp      : 0 kiểu trùng,    4.1 ms
lệch trái 5000 nút, BFS ngược   : 0 kiểu trùng,    2.9 ms
lệch trái 5000 nút, nộp Solution: 0 kiểu trùng,    3.7 ms
zig-zag 5000 nút: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tuần tự hóa bỏ `#` cho nút rỗng

Nếu chỉ ghép giá trị của nút có thật, cây "có con trái 5" và cây "có con phải 5" cho cùng chuỗi, và bị coi là trùng.

In [29]:
root = build_tree([1, 1, 1, 5, None, None, 5])
print("hai nút 1: một nút có con trái 5, một nút có con phải 5")
print("đúng (oracle):", kinds_oracle(root))
print("bản sai       :", sorted(json.dumps(to_list(r)) for r in wrong_no_null(root)))

hai nút 1: một nút có con trái 5, một nút có con phải 5
đúng (oracle): ['[5]']
bản sai       : ['[1, null, 5]', '[5]']


**Vì sao sai:** không có `#`, vị trí (trái hay phải) của con bị mất, nên hai cây **khác cấu trúc** bị coi trùng. **Đúng:** luôn ghi dấu cho nút rỗng, ví dụ `"val,trái,phải"` với `#` cho `None`.

### Lỗi 2 — Quên dấu phân cách giữa các giá trị

In [30]:
root = build_tree([0, 1, 11, 11, None, 1])
print("trái = 1 có con 11; phải = 11 có con 1")
print("đúng (oracle):", kinds_oracle(root))
print("bản sai       :", sorted(json.dumps(to_list(r)) for r in wrong_no_sep(root)))
print("hai chuỗi sai:", "1" + "11##" + "#", "và", "11" + "1##" + "#")

trái = 1 có con 11; phải = 11 có con 1
đúng (oracle): []
bản sai       : ['[11, 1]']
hai chuỗi sai: 111### và 111###


**Vì sao sai:** nếu ghép số thẳng vào nhau thì `1` rồi `11` và `11` rồi `1` đều thành `"111"`. **Đúng:** dùng dấu phân cách (`,`) hoặc đặt giá trị trong ngoặc.

### Lỗi 3 — So sánh nông (chỉ giá trị con trực tiếp)

Khóa `(val, left.val, right.val)` bỏ qua phần cháu chắt trở xuống.

In [31]:
root = build_tree([0, 1, 1, 2, None, 2, None, 9])
print("hai nút 1 cùng có con trái 2, nhưng một nút 2 có con 9, nút kia là lá")
print("đúng (oracle):", kinds_oracle(root))
print("bản sai       :", sorted(json.dumps(to_list(r)) for r in wrong_shallow(root)))

hai nút 1 cùng có con trái 2, nhưng một nút 2 có con 9, nút kia là lá
đúng (oracle): []
bản sai       : ['[1, 2]']


**Vì sao sai:** hai nút `1` trùng ở mức hai tầng đầu nhưng khác nhau ở tầng sâu hơn. **Đúng:** khóa phải đại diện cho **toàn bộ** cây con (chuỗi đầy đủ, hoặc `(val, id_trái, id_phải)` với ID của cây con thật).

### Lỗi 4 — Thêm vào kết quả mỗi lần đếm `>= 2`

In [32]:
root = build_tree(ex1)
print("lá 4 xuất hiện 3 lần")
print("đúng :", kinds_oracle(root))
print("sai  :", sorted(json.dumps(to_list(r)) for r in wrong_ge_two(root)))

lá 4 xuất hiện 3 lần
đúng : ['[2, 4]', '[4]']
sai  : ['[2, 4]', '[4]', '[4]']


**Vì sao sai:** đề yêu cầu **một** đại diện cho mỗi kiểu; ở lần xuất hiện thứ 3 lá `4` lại bị thêm lần nữa. **Đúng:** thêm khi đếm **chạm đúng 2** (`== 2`), hoặc dùng `set` các kiểu đã trả.

### Lỗi 5 — Đệ quy quá sâu với cây lệch

Cách 3 (và cách 2) dùng đệ quy; với cây lệch hơn ~1000 tầng, Python báo `RecursionError`.

In [33]:
deep = make_left_chain(1500)
old_limit = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    find_dup_id_recursive(deep)
    print("không lỗi (bất thường)")
except RecursionError:
    print("RecursionError: cách đệ quy không chịu nổi cây lệch 1500 nút")
finally:
    sys.setrecursionlimit(old_limit)
print("cách lặp trên cùng cây:", find_dup_id_iterative(deep))

RecursionError: cách đệ quy không chịu nổi cây lệch 1500 nút
cách lặp trên cùng cây: []


**Vì sao sai:** mỗi lần gọi đệ quy chiếm một khung trên ngăn xếp; 1500 khung vượt giới hạn 1000. **Cách sửa:** viết lại bằng ngăn xếp tự quản (cách 4) hoặc duyệt BFS ngược (cách 5). (Tăng `sys.setrecursionlimit` cũng là một cách, nhưng không an toàn tuyệt đối vì có thể tràn ngăn xếp C.)

## 14. Mẹo & nhận dạng pattern

- Đề hỏi "cây con/chuỗi con **trùng nhau**" → nghĩ ngay **băm dấu vân tay** rồi đếm.
- Dấu vân tay tốt nhất là **ID nguyên** của `(val, id_trái, id_phải)` (kỹ thuật *hash-consing* / *canonical form*): so sánh `O(1)`, bộ nhớ gọn. Chuỗi tuần tự hóa dễ viết hơn nhưng có thể `O(n^2)`.
- Muốn xử lý cây con trước cha → **hậu thứ tự** (đệ quy) hoặc **BFS rồi duyệt ngược** (không đệ quy).
- Với cây có thể sâu 5000+, luôn nghĩ tới **giới hạn đệ quy của Python**; chuyển sang lặp.
- Đếm bằng `count == 2` để mỗi kiểu chỉ trả một lần (so với `>= 2`).
- Khi thứ tự kết quả không quan trọng, test bằng cách **chuẩn hóa** (tuần tự hóa rồi sort) thay vì so danh sách trực tiếp.
- Tự dựng oracle bằng cách so sánh từng cặp trên cây nhỏ để bắt lỗi tuần tự hóa.

## 15. Biến thể

### 15.1. Đếm số kiểu trùng và tổng số nút nằm trong các cây con trùng

Chỉ cần giữ `count` theo ID; số kiểu trùng = số ID có `count >= 2`.

In [34]:
def count_dup_kinds(root):
    order = [root]
    for node in order:
        if node.left:
            order.append(node.left)
        if node.right:
            order.append(node.right)
    table, sid, count = {}, {None: 0}, Counter()
    for node in reversed(order):
        key = (node.val, sid[node.left], sid[node.right])
        sid[node] = table.setdefault(key, len(table) + 1)
        count[sid[node]] += 1
    kinds = sum(1 for c in count.values() if c >= 2)
    nodes_in_dups = sum(c for c in count.values() if c >= 2)
    return kinds, nodes_in_dups


print(count_dup_kinds(build_tree(ex1)))       # (2, 5): lá 4 (x3) + cây 2->4 (x2)
print(count_dup_kinds(build_tree(ex2)))       # (1, 2)

(2, 5)
(1, 2)


### 15.2. Chỉ so cấu trúc, bỏ qua giá trị

Bỏ `val` khỏi khóa: `key = (sid[left], sid[right])`.

In [35]:
def find_dup_shape(root):
    order = [root]
    for node in order:
        if node.left:
            order.append(node.left)
        if node.right:
            order.append(node.right)
    table, sid, count, ans = {}, {None: 0}, {}, []
    for node in reversed(order):
        key = (sid[node.left], sid[node.right])          # không có val
        ident = sid[node] = table.setdefault(key, len(table) + 1)
        count[ident] = count.get(ident, 0) + 1
        if count[ident] == 2:
            ans.append(node)
    return ans


r = build_tree([5, 1, 2, 3, None, 4])      # hai lá 3 và 4 cùng hình dạng, giá trị khác nhau
print([to_list(x) for x in find_dup_shape(r)])
print([to_list(x) for x in find_dup_id_iterative(r)])

[[3], [1, 3]]
[]


### 15.3. Subtree of Another Tree (LeetCode 572)

Cây `subRoot` có là cây con của `root` không? Gán ID **chung một bảng** cho cả hai cây rồi kiểm tra `ID(subRoot)` có trong tập ID các nút của `root`.

In [36]:
def is_subtree(root, sub):
    table = {}

    def ids_of(r):
        order = [r]
        for node in order:
            if node.left:
                order.append(node.left)
            if node.right:
                order.append(node.right)
        sid = {None: 0}
        for node in reversed(order):
            key = (node.val, sid[node.left], sid[node.right])
            sid[node] = table.setdefault(key, len(table) + 1)
        return sid

    sid_root = ids_of(root)
    sid_sub = ids_of(sub)
    return sid_sub[sub] in {v for k, v in sid_root.items() if k is not None}


t = build_tree([3, 4, 5, 1, 2])
print(is_subtree(t, build_tree([4, 1, 2])))                  # True
print(is_subtree(t, build_tree([4, 1, 2, None, None, 0])))   # False

True
False


### 15.4. Trùng nhau "sau khi lật" (liên quan LeetCode 951 — Flip Equivalent Binary Trees)

Coi hai cây con là trùng nếu có thể hoán đổi con trái/phải ở một số nút. Dùng khóa chuẩn tắc `(val, min(id_trái, id_phải), max(id_trái, id_phải))`.

In [37]:
def find_dup_flip(root):
    order = [root]
    for node in order:
        if node.left:
            order.append(node.left)
        if node.right:
            order.append(node.right)
    table, sid, count, ans = {}, {None: 0}, {}, []
    for node in reversed(order):
        l, r = sid[node.left], sid[node.right]
        key = (node.val, min(l, r), max(l, r))
        ident = sid[node] = table.setdefault(key, len(table) + 1)
        count[ident] = count.get(ident, 0) + 1
        if count[ident] == 2:
            ans.append(node)
    return ans


# 7->(3 trái) và 7->(3 phải) là "lật" của nhau
r = build_tree([0, 7, 7, 3, None, None, 3])
print("không lật:", [to_list(x) for x in find_dup_id_iterative(r)])
print("có lật   :", [to_list(x) for x in find_dup_flip(r)])

không lật: [[3]]
có lật   : [[3], [7, 3]]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. So sánh từng cặp | `same(a, b)` cho mọi cặp cây con | `O(n^3)` | `O(n)` | Oracle kiểm tra, `n` rất nhỏ |
| 2. Tuần tự hóa chuỗi | Khóa = chuỗi `"val,trái,phải"` + `Counter` | `O(n^2)` tệ nhất | `O(n^2)` | Dễ viết, `n` nhỏ/cây cân bằng (đây là lời giải của doocs) |
| 3. ID đệ quy | Khóa = `(val, id_trái, id_phải)` | `O(n)` | `O(n)` + ngăn xếp | Ngắn gọn khi cây không quá sâu |
| 4. ID hậu thứ tự lặp | Cùng khóa, ngăn xếp tự quản | `O(n)` | `O(n)` | **Lời giải nộp**, an toàn với cây lệch |
| 5. ID BFS ngược | Duyệt BFS, xử lý ngược để con trước cha | `O(n)` | `O(n)` | Không đệ quy, code ngắn, dễ nghĩ |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print(serialize(build_tree([1, 2, 3])))
print(serialize(build_tree([1, None, 2])))
print([to_list(r) for r in Solution().findDuplicateSubtrees(build_tree([5, 5, 5]))])
```

In [38]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `root = [2,2,2,3,null,3,null]` (ví dụ 3), hãy điền bảng ID hậu thứ tự: với mỗi nút theo thứ tự trái → phải → cha, ghi `key`, `ID`, `count`. Hỏi: có bao nhiêu kiểu trùng, và các kiểu đó là gì?

In [39]:
# Bài 2: điền kết quả tính tay
hand_keys_in_order = None     # list các key (val, id_trái, id_phải) theo thứ tự xử lý (cây có 5 nút)
hand_num_kinds = None
hand_kinds = None             # list các cây con dạng danh sách theo tầng, ví dụ [[1]]

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành lời giải dùng **chuỗi tuần tự hóa** nhưng với `set` các kiểu đã trả (thay vì `== 2`).

In [40]:
def exercise_3_serialize_set(root):
    seen = set()
    reported = set()
    ans = []

    def dfs(node):
        # TODO 1: nếu node là None trả "#"
        # TODO 2: v = chuỗi "val,trái,phải" (gọi dfs cho hai con)
        # TODO 3: nếu v đã nằm trong seen và chưa nằm trong reported -> thêm node vào ans, thêm v vào reported
        # TODO 4: thêm v vào seen; trả v
        return None

    # TODO 5: gọi dfs(root) rồi trả ans
    return None


# assert sorted(json.dumps(to_list(r)) for r in exercise_3_serialize_set(build_tree(ex1))) == sorted(["[2, 4]", "[4]"])
# assert [to_list(r) for r in exercise_3_serialize_set(build_tree(ex2))] == [[1]]
# assert exercise_3_serialize_set(build_tree([1])) == []

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại cách 3, hãy tự viết `exercise_4_ids(root)` gán ID **đệ quy** với khóa `(val, id_trái, id_phải)`; trả về danh sách nút đại diện cho các kiểu trùng.

In [41]:
def exercise_4_ids(root):
    # TODO
    pass


# assert sorted(json.dumps(to_list(r)) for r in exercise_4_ids(build_tree(ex1))) == sorted(["[2, 4]", "[4]"])
# assert [to_list(r) for r in exercise_4_ids(build_tree(ex2))] == [[1]]
# assert sorted(json.dumps(to_list(r)) for r in exercise_4_ids(build_tree(ex3))) == sorted(["[2, 3]", "[3]"])

### Bài 5 — Mini challenge: kiểu xuất hiện ít nhất 3 lần (khó)

Viết `exercise_5_at_least_three(root)` trả về **một đại diện** cho mỗi kiểu cây con xuất hiện **ít nhất 3 lần**. Gợi ý: đếm theo ID và thêm vào kết quả khi `count` chạm đúng 3.

```python
exercise_5_at_least_three(build_tree(ex1))  -> [[4]]        # lá 4 xuất hiện 3 lần; cây 2->4 chỉ 2 lần
exercise_5_at_least_three(build_tree([0] * 7))  -> [[0]]    # 4 lá 0 giống nhau; hai cây 0->(0,0) chỉ 2 lần
```

In [42]:
def exercise_5_at_least_three(root):
    # TODO
    pass


# assert [to_list(r) for r in exercise_5_at_least_three(build_tree(ex1))] == [[4]]
# assert [to_list(r) for r in exercise_5_at_least_three(build_tree([0] * 7))] == [[0]]
# assert exercise_5_at_least_three(build_tree(ex2)) == []

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: Tiền thứ tự `"val,trái,phải"`, `#` cho `None`.
- Mức 2: `[1,2,3]` là gốc 1, con trái 2, con phải 3 (cả hai lá).
- Mức 3: `1,2,#,#,3,#,#`; `1,#,2,#,#`; `[[5]]`.

**Bài 2**
- Mức 1: Cây có 5 nút: gốc 2, hai nút 2, hai lá 3.
- Mức 2: Thứ tự hậu thứ tự: lá `3` (dưới 2 trái), `2` trái, lá `3` (dưới 2 phải), `2` phải, gốc `2`.
- Mức 3: Xem đáp án mục 19.

**Bài 3**
- Mức 1: Dùng hai tập: `seen` (đã gặp) và `reported` (đã trả).
- Mức 2: Chỉ thêm vào `ans` khi `v in seen and v not in reported`.
- Mức 3: Xem đáp án mục 19.

**Bài 4**
- Mức 1: `table = {}`, `count = {}`, hàm con `dfs(node)` trả ID.
- Mức 2: `ident = table.setdefault(key, len(table) + 1)`.
- Mức 3: Xem `find_dup_id_recursive` ở mục 7.2.

**Bài 5**
- Mức 1: Giống cách 3, nhưng ngưỡng đếm là 3.
- Mức 2: Điều kiện `count[ident] == 3`.
- Mức 3: Xem đáp án mục 19.

## 19. Đáp án tham khảo

### Đáp án Bài 1

In [43]:
print(serialize(build_tree([1, 2, 3])))                       # 1,2,#,#,3,#,#
print(serialize(build_tree([1, None, 2])))                    # 1,#,2,#,#
print([to_list(r) for r in Solution().findDuplicateSubtrees(build_tree([5, 5, 5]))])   # [[5]]

assert serialize(build_tree([1, 2, 3])) == "1,2,#,#,3,#,#"
assert serialize(build_tree([1, None, 2])) == "1,#,2,#,#"
assert [to_list(r) for r in Solution().findDuplicateSubtrees(build_tree([5, 5, 5]))] == [[5]]

1,2,#,#,3,#,#
1,#,2,#,#
[[5]]


### Đáp án Bài 2

Thứ tự hậu thứ tự: lá `3` (dưới `2` trái) → `2` trái → lá `3` (dưới `2` phải) → `2` phải → gốc `2`.

| Nút | `key` | ID | `count` |
|---|---|---|---|
| lá `3` (trái) | `(3,0,0)` | 1 | 1 |
| `2` trái | `(2,1,0)` | 2 | 1 |
| lá `3` (phải) | `(3,0,0)` | 1 | **2** |
| `2` phải | `(2,1,0)` | 2 | **2** |
| gốc `2` | `(2,2,2)` | 3 | 1 |

Có **2 kiểu**: `[3]` và `[2, 3]`.

In [44]:
hand_keys_in_order = [(3, 0, 0), (2, 1, 0), (3, 0, 0), (2, 1, 0), (2, 2, 2)]
hand_num_kinds = 2
hand_kinds = [[3], [2, 3]]

# kiểm tra bằng code: dựng ID theo đúng thứ tự trên
root = build_tree(ex3)
table, sid, keys = {}, {None: 0}, []
stack = [root]
while stack:
    node = stack[-1]
    if node.left is not None and node.left not in sid:
        stack.append(node.left)
        continue
    if node.right is not None and node.right not in sid:
        stack.append(node.right)
        continue
    stack.pop()
    key = (node.val, sid[node.left], sid[node.right])
    keys.append(key)
    sid[node] = table.setdefault(key, len(table) + 1)
print(keys)
assert keys == hand_keys_in_order
res = Solution().findDuplicateSubtrees(build_tree(ex3))
assert len(res) == hand_num_kinds
assert sorted(json.dumps(to_list(r)) for r in res) == sorted(json.dumps(k) for k in hand_kinds)
print("Bài 2 OK")

[(3, 0, 0), (2, 1, 0), (3, 0, 0), (2, 1, 0), (2, 2, 2)]
Bài 2 OK


### Đáp án Bài 3

In [45]:
def exercise_3_serialize_set_solution(root):
    seen = set()
    reported = set()
    ans = []

    def dfs(node):
        if node is None:
            return "#"
        v = f"{node.val},{dfs(node.left)},{dfs(node.right)}"
        if v in seen and v not in reported:
            ans.append(node)
            reported.add(v)
        seen.add(v)
        return v

    dfs(root)
    return ans


assert sorted(json.dumps(to_list(r)) for r in exercise_3_serialize_set_solution(build_tree(ex1))) == sorted(["[2, 4]", "[4]"])
assert [to_list(r) for r in exercise_3_serialize_set_solution(build_tree(ex2))] == [[1]]
assert exercise_3_serialize_set_solution(build_tree([1])) == []
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [46]:
def exercise_4_ids_solution(root):
    table, count, ans = {}, {}, []

    def dfs(node):
        if node is None:
            return 0
        key = (node.val, dfs(node.left), dfs(node.right))
        ident = table.setdefault(key, len(table) + 1)
        count[ident] = count.get(ident, 0) + 1
        if count[ident] == 2:
            ans.append(node)
        return ident

    dfs(root)
    return ans


assert sorted(json.dumps(to_list(r)) for r in exercise_4_ids_solution(build_tree(ex1))) == sorted(["[2, 4]", "[4]"])
assert [to_list(r) for r in exercise_4_ids_solution(build_tree(ex2))] == [[1]]
assert sorted(json.dumps(to_list(r)) for r in exercise_4_ids_solution(build_tree(ex3))) == sorted(["[2, 3]", "[3]"])
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [47]:
def exercise_5_at_least_three_solution(root):
    table, count, ans = {}, {}, []
    order = [root]
    for node in order:
        if node.left:
            order.append(node.left)
        if node.right:
            order.append(node.right)
    sid = {None: 0}
    for node in reversed(order):
        key = (node.val, sid[node.left], sid[node.right])
        ident = sid[node] = table.setdefault(key, len(table) + 1)
        count[ident] = count.get(ident, 0) + 1
        if count[ident] == 3:
            ans.append(node)
    return ans


assert [to_list(r) for r in exercise_5_at_least_three_solution(build_tree(ex1))] == [[4]]
assert [to_list(r) for r in exercise_5_at_least_three_solution(build_tree([0] * 7))] == [[0]]
assert exercise_5_at_least_three_solution(build_tree(ex2)) == []

# đối chiếu với đếm vét cạn trên cây ngẫu nhiên
rng = random.Random(55)
for _ in range(200):
    tree = random_tree(rng.randint(1, 40), rng, vmax=1)
    nodes = all_nodes(tree)
    kinds = {}
    for x in nodes:
        k = serialize(x)
        kinds[k] = kinds.get(k, 0) + 1
    expected = sorted(json.dumps(to_list(x)) for x in
                      {serialize(y): y for y in nodes if kinds[serialize(y)] >= 3}.values())
    got = sorted(json.dumps(to_list(r)) for r in exercise_5_at_least_three_solution(tree))
    assert got == expected
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (gán ID, hậu thứ tự lặp)

```python
class Solution:
    def findDuplicateSubtrees(self, root: Optional[TreeNode]) -> List[Optional[TreeNode]]:
        if root is None:
            return []
        table, count, ans = {}, {}, []
        sid = {None: 0}                           # nút -> ID dạng cây con
        stack = [root]
        while stack:
            node = stack[-1]
            if node.left is not None and node.left not in sid:
                stack.append(node.left); continue
            if node.right is not None and node.right not in sid:
                stack.append(node.right); continue
            stack.pop()
            key = (node.val, sid[node.left], sid[node.right])
            ident = table.setdefault(key, len(table) + 1)
            sid[node] = ident
            count[ident] = count.get(ident, 0) + 1
            if count[ident] == 2:
                ans.append(node)
        return ans
```

Bản đệ quy ngắn (khi cây không sâu): `key = (node.val, dfs(node.left), dfs(node.right))` rồi cấp ID và đếm như trên.

### Checklist trước khi submit

- [ ] Khóa đại diện cho **toàn bộ** cây con: `(val, id_trái, id_phải)` hoặc chuỗi có `#` và dấu phân cách.
- [ ] Cây rỗng có ID cố định (`0`), mọi dạng khác ID `>= 1`.
- [ ] Thêm vào kết quả khi `count == 2` (mỗi kiểu đúng một lần).
- [ ] Con xử lý trước cha (hậu thứ tự hoặc duyệt ngược BFS).
- [ ] Không đệ quy sâu với cây lệch 5000 nút (dùng ngăn xếp), hoặc ý thức rõ giới hạn.
- [ ] Test không phụ thuộc thứ tự trả về: chuẩn hóa bằng tuần tự hóa rồi sort.
- [ ] Signature đúng: `class Solution`, `findDuplicateSubtrees(self, root: Optional[TreeNode]) -> List[Optional[TreeNode]]`.